# EinsteinEngine vs. Pure SymPy: The Exact Kerr Black Hole Benchmark

This benchmark demonstrates the true computational superiority of **EinsteinEngine**. 

While abstract functions (`sp.Function`) bypass heavy algebraic simplification, **concrete explicit metrics** force the symbolic engines to actively compute quotient rules, chain rules, polynomial expansions, and massive common denominators.

### The Ultimate Test: The Kerr Metric
We use the exact explicit metric for a rotating Black Hole in Boyer-Lindquist coordinates. It features extreme fractional coupling ($\Delta$, $\rho^2$), trigonometric functions, and off-diagonal frame-dragging terms. 

Because Kerr is a vacuum solution, after expanding and contracting tens of thousands of intermediate tensor terms, the final **Ricci Scalar MUST exactly equal 0**. We will test which engine can collapse this mathematical monster the fastest.

In [ ]:
import sys
from pathlib import Path

def _find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "einsteinengine").exists():
            return candidate
    return start

REPO_ROOT = _find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import time
import symengine as se
import sympy as sp

# 1. Setup explicit symbols
t, r, theta, phi = sp.symbols('t r theta phi')
M, a = sp.symbols('M a') # Mass and Spin parameters
syms = [t, r, theta, phi]

# 2. Define the explicit Kerr metric structural functions
rho2 = r**2 + a**2 * sp.cos(theta)**2
Delta = r**2 - 2*M*r + a**2

# 3. Define the exact non-diagonal matrix
g_kerr = [
    [-(1 - 2*M*r/rho2), 0, 0, -2*M*a*r*sp.sin(theta)**2 / rho2],
    [0, rho2/Delta, 0, 0],
    [0, 0, rho2, 0],
    [-2*M*a*r*sp.sin(theta)**2 / rho2, 0, 0, (r**2 + a**2 + 2*M*a**2*r*sp.sin(theta)**2 / rho2)*sp.sin(theta)**2]
]

## 1. EinsteinEngine Execution
Leveraging C++ pointers (SymEngine) and the `einsum` on-the-fly contraction architecture, EinsteinEngine dynamically reduces fractions and prevents combinatorial explosion in memory.

In [2]:
from einsteinengine.symbolic.metric import MetricTensor
from einsteinengine.symbolic.riemann import RiemannCurvatureTensor
from einsteinengine.symbolic.ricci_tensor import RicciTensor
from einsteinengine.symbolic.ricci_scalar import RicciScalar

print("--- Running EinsteinEngine (C++ Optimized Engine) ---")
start_ee = time.time()

metric_ee = MetricTensor(g_kerr, syms, name="KerrExact", verbose=False)
riemann_ee = RiemannCurvatureTensor.from_metric(metric_ee, verbose=False)
ricci_tensor_ee = RicciTensor.from_riemann(riemann_ee, verbose=False)
ricci_scalar_ee = RicciScalar.from_ricci_tensor_and_metric(ricci_tensor_ee, metric_ee, verbose=False)

end_ee = time.time()
ee_time = end_ee - start_ee

print(f"EinsteinEngine Execution Time: {ee_time:.4f} seconds")

--- Running EinsteinEngine (C++ Optimized Engine) ---
EinsteinEngine Execution Time: 4.2958 seconds


## 2. Pure SymPy Execution
Using standard SymPy and Python loops. To prevent SymPy from completely crashing the RAM with millions of unsimplified nodes, we apply `.simplify()` at the end of the calculation to force it to find the vacuum `0`.

In [3]:
print("--- Running Pure SymPy (Standard Python Engine) ---")
start_sp = time.time()

g_sp = sp.Matrix(g_kerr)
g_inv_sp = g_sp.inv() # This inversion alone generates massive algebraic trees

# Christoffel Symbols
Gamma_sp = [[[0 for _ in range(4)] for _ in range(4)] for _ in range(4)]
for lam in range(4):
    for mu in range(4):
        for nu in range(4):
            tmp_sum = 0
            for sigma in range(4):
                term1 = sp.diff(g_sp[nu, sigma], syms[mu])
                term2 = sp.diff(g_sp[mu, sigma], syms[nu])
                term3 = sp.diff(g_sp[mu, nu], syms[sigma])
                tmp_sum += g_inv_sp[lam, sigma] * (term1 + term2 - term3)
            Gamma_sp[lam][mu][nu] = sp.Rational(1, 2) * tmp_sum

# Riemann Curvature Tensor
R_riemann_sp = [[[[0 for _ in range(4)] for _ in range(4)] for _ in range(4)] for _ in range(4)]
for rho in range(4):
    for sigma in range(4):
        for mu in range(4):
            for nu in range(4):
                term1 = sp.diff(Gamma_sp[rho][nu][sigma], syms[mu])
                term2 = sp.diff(Gamma_sp[rho][mu][sigma], syms[nu])
                term3 = sum(Gamma_sp[rho][mu][lam] * Gamma_sp[lam][nu][sigma] for lam in range(4))
                term4 = sum(Gamma_sp[rho][nu][lam] * Gamma_sp[lam][mu][sigma] for lam in range(4))
                R_riemann_sp[rho][sigma][mu][nu] = term1 - term2 + term3 - term4

# Ricci Tensor
Ricci_sp = [[0 for _ in range(4)] for _ in range(4)]
for a in range(4):
    for b in range(4):
        Ricci_sp[a][b] = sum(R_riemann_sp[lam][a][lam][b] for lam in range(4))

# Ricci Scalar
R_scalar_sp = 0
for mu in range(4):
    for nu in range(4):
        R_scalar_sp += g_inv_sp[mu, nu] * Ricci_sp[mu][nu]

# Force mathematical collapse to prove it reaches 0
R_scalar_sp = sp.simplify(R_scalar_sp)

end_sp = time.time()
sp_time = end_sp - start_sp

print(f"Pure SymPy Execution Time: {sp_time:.4f} seconds")

--- Running Pure SymPy (Standard Python Engine) ---


KeyboardInterrupt: 

## 3. Final Performance Summary

In [ ]:
sp_time=30*60 # Stopped the previous cell at 30 minutes since sympy couldn't resolve it

print("========================================")
print("          BENCHMARK SUMMARY             ")
print("========================================")
print(f"EinsteinEngine Execution Time : {ee_time:.4f} s")
print(f"Pure SymPy Execution Time     : {sp_time:.4f} s")
print("----------------------------------------")
if ee_time > 0:
    speedup = sp_time / ee_time
    print(f"PERFORMANCE MULTIPLIER: {speedup:.2f}x FASTER!")
print("========================================")

print("\nMathematical Verification (Should be exactly 0 for a Kerr Vacuum):")
print(f"EE Ricci Scalar: {ricci_scalar_ee.get_raw_data()}")
print(f"SP Ricci Scalar: {R_scalar_sp}")

          BENCHMARK SUMMARY             
EinsteinEngine Execution Time : 4.2958 s
Pure SymPy Execution Time     : 1800.0000 s
----------------------------------------
PERFORMANCE MULTIPLIER: 419.01x FASTER!

Mathematical Verification (Should be exactly 0 for a Kerr Vacuum):
EE Ricci Scalar: (-(-2*M*r + a**2 + r**2)/(a**2*cos(theta)**2 + r**2) + a**2*sin(theta)**2/(a**2*cos(theta)**2 + r**2) - a**2*cos(theta)**2/(a**2*cos(theta)**2 + r**2) + r**2*(-2*M*r + a**2 + r**2)/(a**2*cos(theta)**2 + r**2)**2 - (-2*M + 2*r)*r/(a**2*cos(theta)**2 + r**2) + (-1/2)*r*(-2*M*r + a**2 + r**2)*((2*M/(a**2*cos(theta)**2 + r**2) - 4*M*r**2/(a**2*cos(theta)**2 + r**2)**2)*(-a**2*r**2 - a**4*cos(theta)**2 - a**2*r**2*cos(theta)**2 - 2*M*a**2*r*sin(theta)**2 - r**4)/(-2*M*r**3 + a**2*r**2 + a**4*cos(theta)**2 - 2*M*a**2*r + a**2*r**2*cos(theta)**2 + 2*M*a**2*r*sin(theta)**2 + r**4) - 2*M*a*r*(-2*M*a*sin(theta)**2/(a**2*cos(theta)**2 + r**2) + 4*M*a*r**2*sin(theta)**2/(a**2*cos(theta)**2 + r**2)**2)/(-2*M*r**